# Exercício 2 - CRUD - Controle de Estoque - Adicionando Insumo (CREATE) <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/microsoftsqlserver/microsoftsqlserver-plain.svg" height="45" />➕

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![pyodbc](https://img.shields.io/badge/pyodbc-0078D4?style=flat-square)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-exercício%20%7C%20crud%20%7C%20estoque-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Intermediário-yellow)
![Pré-req](https://img.shields.io/badge/pré--req-pyodbc%20%7C%20funções-purple)
![Biblioteca](https://img.shields.io/badge/requer-pyodbc-orange)

> A função `adicionar_insumo()`: cadastra um insumo novo — mas se ele **já existir**, soma a quantidade em vez de criar uma linha duplicada. É a diferença entre um `INSERT` ingênuo e um CREATE que respeita a regra de negócio (um insumo, uma linha).

## 📋 Conteúdo

1. [Conectando](#-1-conectando)
2. [Por que Não Só um INSERT](#-2-por-que-não-só-um-insert)
3. [Construindo adicionar_insumo()](#-3-construindo-adicionar_insumo)
4. [Testando a Função](#-4-testando-a-função)


## 🔌 1. Conectando

In [1]:
from conexao import nova_conexao_sqlserver
from cores import *

conexao = nova_conexao_sqlserver(banco="HashtagCursoSQL", autocommit=True)
cursor = conexao.cursor()


## 🤔 2. Por que Não Só um INSERT

Se `Farinha de Trigo` chegar de novo (uma nova compra), um `INSERT` simples criaria uma segunda linha — e a coluna `UNIQUE(Insumo)` do notebook anterior bloquearia isso com erro. O comportamento certo é: **se o insumo já existe, soma a quantidade recebida na que já tem.**

## 🛠️ 3. Construindo adicionar_insumo()

A função primeiro checa se o insumo existe (`SELECT`); se existir, faz `UPDATE` somando a quantidade; se não existir, faz `INSERT`.

In [2]:
def adicionar_insumo(nome, quantidade, unidade_medida=None, categoria=None):
    cursor.execute("SELECT Id, Quantidade FROM dbo.Estoque WHERE Insumo = ?", nome)
    insumo_existente = cursor.fetchone()

    if insumo_existente:
        nova_quantidade = float(insumo_existente.Quantidade) + quantidade
        cursor.execute(
            "UPDATE dbo.Estoque SET Quantidade = ?, AtualizadoEm = GETDATE() WHERE Id = ?",
            nova_quantidade, insumo_existente.Id
        )
        print(f"{CinzaClaro}{nome}{Reset} já existia — {VerdeClaro}+{quantidade}{Reset}, total agora: {MagentaClaro}{nova_quantidade}{Reset}")
    else:
        cursor.execute(
            """INSERT INTO dbo.Estoque (Insumo, Quantidade, UnidadeMedida, Categoria)
               VALUES (?, ?, ?, ?)""",
            nome, quantidade, unidade_medida, categoria
        )
        print(f"{VerdeClaro}{nome} cadastrado{Reset} — quantidade inicial: {MagentaClaro}{quantidade}{Reset}")


## 🧪 4. Testando a Função

Um caso de insumo que já existe (soma) e um caso de insumo novo (cadastra).

In [3]:
adicionar_insumo("Farinha de Trigo", 20.0)          # já existe -> soma
adicionar_insumo("Fermento Químico", 6.0, "kg", "Grãos")  # novo -> cadastra

print(f"\n{CinzaClaro}Estoque atualizado:{Reset}")
cursor.execute("SELECT Insumo, Quantidade, UnidadeMedida FROM dbo.Estoque ORDER BY Insumo")
for linha in cursor.fetchall():
    print(f"  {VerdeClaro}{linha.Insumo}{Reset} — {linha.Quantidade} {linha.UnidadeMedida}")

conexao.close()


Farinha de Trigo já existia — +20.0, total agora: 70.0
Fermento Químico cadastrado — quantidade inicial: 6.0

Estoque atualizado:
  Açúcar — 40.00 kg
  Chocolate em Pó — 12.00 kg
  Farinha de Trigo — 70.00 kg
  Fermento Biológico — 5.00 kg
  Fermento Químico — 6.00 kg
  Leite — 80.00 L
  Manteiga — 15.00 kg
  Morango — 8.00 kg
  Ovos — 300.00 un


`adicionar_insumo()` já resolve o "C" do CRUD com a regra de negócio certa. O próximo notebook cuida do extremo oposto: remover um insumo do controle.

> ▶️ Próximo notebook: **Exercício 2 - CRUD - Controle de Estoque - Deletar Insumo (DELETE)**.